# Part 3: Production Pipeline Deployment & Self-Evaluation Dashboard
This notebook imports your decoupled `src/` module package objects, runs all 5 core clinical problem statements, initiates the automated self-audit check, and exports the final metrics workbook directly to disk.

In [ ]:
import sys
import os
import pandas as pd

# Ensure source paths link properly to the python sys framework variables
sys.path.append(os.path.abspath("../"))

from src.core_pipeline import MedicalRAGApp
from src.evaluation_judge import ClinicalRAGJudge
print("✓ App classes and automated audit modules loaded successfully.")

In [ ]:
# Initialize production objects (Assumes your 'llm' variable is running in memory)
app = MedicalRAGApp(chroma_dir="../data/chroma_db")
judge = ClinicalRAGJudge()

# Structured target query payload definitions
target_questions = [
    {"category": "Critical Care Protocols", "query": "What is the protocol for managing sepsis in a critical care unit?"},
    {"category": "Diagnostic Assistance", "query": "What are the common symptoms and treatments for pulmonary embolism?"},
    {"category": "Drug Information", "query": "Can you provide the trade names of medications used for treating hypertension?"},
    {"category": "Treatment Plans", "query": "What are the first-line options and alternatives for managing rheumatoid arthritis?"},
    {"category": "Specialty Knowledge", "query": "What are the diagnostic steps for suspected endocrine disorders?"}
]

report_payloads = []

In [ ]:
print("Starting production-grade evaluation loop...")
for index, item in enumerate(target_questions, start=1):
    cat = item["category"]
    qry = item["query"]
    
    # 1. Generate RAG output utilizing strict production configuration
    pipeline_data = app.generate_response(llm_instance=llm, query=qry, category_filter=cat, k=4)
    
    # 2. Fire Judge validation suite
    audit_results = judge.audit_response(
        llm_instance=llm, 
        query=qry, 
        context=pipeline_data["context"], 
        answer=pipeline_data["answer"]
    )
    
    report_payloads.append({
        "Query_ID": f"Q_{index}",
        "Category": cat,
        "Query": qry,
        "RAG_Answer": pipeline_data["answer"],
        "Groundedness_Report": audit_results["groundedness"],
        "Relevance_Report": audit_results["relevance"]
    })
    print(f"✓ Completed Query [{index}/5]")

In [ ]:
# Compile evaluations dataframe and commit write sequence to disk
df_final_report = pd.DataFrame(report_payloads)
output_path = "../reports/clinical_rag_evaluation_report.csv"
df_final_report.to_csv(output_path, index=False, encoding='utf-8')

print(f"🎉 Complete! Pipeline report written to path: '{output_path}'")
df_final_report[["Query_ID", "Category", "Groundedness_Report", "Relevance_Report"]].head()